In [ ]:
using Base.Threads
println( "Number of threads: ", nthreads() )

include( "../args.jl" )
include( "../geom.jl" )
include( "../recur.jl" )

base = "../data/results/";

In [ ]:
# Case title.
N = 100;  μ = 1.0;  ϕ = 1/4
ξ = 1.0;  γ = 0.01;  s = 10.0

# Temporal variables.
T = 5000
tlist = 0:5:T

# Initialize β list.
βmin = -2;  βmax = 0
Nβ = 51;  Δβ = (βmax - βmin)/(Nβ-1)
βlist = round.( 10.0.^(βmin:Δβ:βmax), digits=6 )  # NON-DIMENSIONAL

# Initialize τ list.
τmin = 500;  τmax = 1000
Nτ = 51;  Δτ = (τmax - τmin)/(Nτ-1)
τlist = round.( τmin:Δτ:τmax, digits=6 )  # NON-DIMENSIONAL

# Number of parameter combinations.
M = 10
println( "Important simulation data for ($(Nβ),$(Nτ)) unique parameter cases." )

# Adapatable time-step length.
ρmin = -4;  ρmax = 0
Nρ = 51;  Δρ = (ρmax - ρmin)/(Nρ-1)
ρlist = round.( 10.0.^(ρmin:Δρ:ρmax), digits=6 )  # NON-DIMENSIONAL
println( "Each parameter case contains $(Nρ) values for ρ and $(M) replicates." )

In [ ]:
# Create parameter sets.
nondim1data = [[Nondim(; ρ=ρ, β=β, s=s ) for ρ ∈ ρlist] for β ∈ βlist]
nondim2data = [[Nondim(; ρ=ρ, τ=τ, s=s ) for ρ ∈ ρlist] for τ ∈ τlist]

# Critical parameter conditions.
ρc1list = [criticalρ( nondimlist[1], N, μ ) for nondimlist ∈ nondim1data]
ρc2list = [criticalρ( nondimlist[1], N, μ ) for nondimlist ∈ nondim2data];

In [ ]:
# Data folder name.
folder1data = [[findfolder( N, μ, nondim; base=base ) for nondim ∈ nondimlist] for nondimlist ∈ nondim1data]
folder2data = [[findfolder( N, μ, nondim; base=base ) for nondim ∈ nondimlist] for nondimlist ∈ nondim2data];

In [ ]:
# Import activity and determinism data.
ς1datalist = [hcat( [readdlm( folder*"determinism_T-$(round( defInt, T ))_M-$(M).txt" )
    for folder ∈ folderlist]... ) for folderlist ∈ folder1data];
ς2datalist = [hcat( [readdlm( folder*"determinism_T-$(round( defInt, T ))_M-$(M).txt" )
    for folder ∈ folderlist]... ) for folderlist ∈ folder2data];

# Determinism statistics.
ς̄1data = [vcat( mean( ςdata, dims=1 )... ) for ςdata ∈ ς1datalist]
ς̄2data = [vcat( mean( ςdata, dims=1 )... ) for ςdata ∈ ς2datalist];

In [ ]:
# Plot the determinism in each case for visual inspection.
plt = plot( size=(300,200), dpi=100 )

for (k, ς̄list) ∈ enumerate( ς̄1data )
    plot!( plt, ρlist, ς̄list; lw=2, marker=:circ, label="" )
end

plot!( plt; xlims=(ρlist[1],ρlist[end]), xscale=:log10 )
plot!( plt; ylims=(0,1) )

plot!( plt; xlabel="spontaneous deactivation rate, "*L"ρ", ylabel="determinism", legend=:outerright )

In [ ]:
# Critical ρ in default case..
N̂list = [50, 100, 500, 1000];  NN = length( N̂list );  M̂ = 50
ρclist = [criticalρ( Nondim(), N̂, μ ) for N̂ ∈ N̂list];

# Import default determinism example.
ρ̂list = Vector{defFloat}( 10.0.^(-4:0.25:0) )  # NON-DIMENSIONAL
ς̂datalist = [hcat( [
    readdlm( findfolder( N̂, μ, Nondim(; ρ=ρ, s=s ); base=base )*"determinism_T-$(round( defInt, T ))_M-$(M̂).txt" )
    for ρ ∈ ρ̂list]... ) for N̂ ∈ N̂list]

# Compute mean and standard devation.
μς̂data = hcat( [mean( ς̂data, dims=1 )' for ς̂data ∈ ς̂datalist]... )
σς̂data = hcat( [std(  ς̂data, dims=1 )' for ς̂data ∈ ς̂datalist]... );

In [ ]:
colorlist = [:black :cornflowerblue :indianred :mediumpurple :olivedrab]
cmap = cgrad( :ice, [1, 2]./3, rev=false )
llist = hcat( [latexstring( "N=$(N̂)" ) for N̂ ∈ N̂list]... )

# Initialize plots.
plt = plot( layout=(@layout [a{0.45w} b c]), size=(950,250), dpi=100 )
plot!( plt; left_margin=15pt, bottom_margin=25pt, right_margin=15pt )

# Plot determinism example.
plot!( plt[1], [ρclist[1], ρclist[1]], [0, 1]; color=:gray68, lw=3, label="" )
plot!( plt[1], ρ̂list, μς̂data; ribbon=σς̂data, color=colorlist, fillalpha=[1/4 1/2], lw=2, marker=:circ,
    label=hcat( [latexstring( "N=$(N̂)" ) for N̂ ∈ N̂list]... ) )

plot!( plt[1]; xlims=10.0.^[-4,0], xscale=:log10 )
plot!( plt[1]; ylims=(0,1) )

plot!( plt[1]; xlabel="spontaneous deactivation rate, "*L"ρ", ylabel="determinism, "*L"ς", legend=:topleft )
title!( plt[1], "(a)"; title_position=:left )

# Plot heatmap of determinism for changing β.
heatmap!( plt[2], ρlist, βlist, hcat( ς̄1data... )'; cmap=cmap, clims=(0,1), colorbar=false )
plot!( plt[2], ρc1list, βlist; color=:gray68, lw=3, label="critical "*L"ρ" )

plot!( plt[2]; xlims=(ρlist[1],ρlist[end]), xticks=10.0.^(-4:2:0), xscale=:log10 )
plot!( plt[2]; ylims=(βlist[1],βlist[end]), yticks=10.0.^(-2:1:0), yscale=:log10 )

plot!( plt[2]; xlabel="spontaneous deactivation rate, "*L"ρ", ylabel="social activation rate, "*L"β", legend=:topleft )
title!( plt[2], "(b)"; title_position=:left )

# Plot heatmap of determinism for changing τ.
heatmap!( plt[3], ρlist, τlist, hcat( ς̄2data... )'; cmap=cmap, clims=(0,1), colorbar=false )
plot!( plt[3], ρc2list, τlist; color=:gray68, lw=3, label="" )

plot!( plt[3]; xlims=(ρlist[1],ρlist[end]), xticks=10.0.^(-4:2:0), xscale=:log10 )
plot!( plt[3]; ylims=(τlist[1],τlist[end]), yticks=[500,750,1000], yscale=:linear )

plot!( plt[3]; xlabel="spontaneous deactivation rate, "*L"ρ", ylabel="refractory delay, "*L"τ" )
title!( plt[3], "(c)"; title_position=:left )

# saveplot( plt, figurefolder*"determinism-vs-rho_beta-tau_N-$(N̂list[1])-$(N̂list[end]).png"; background=:white, dpi=600 );

In [ ]:
# Create dummy gradient data.
n = 1000
vals = reshape( range(0, 1, length=n), :, 1 )

# Plot the gradient to make colorbar.
plt = plot( size=(100,250), dpi=100 )

plot!( plt; left_margin=7.5pt, right_margin=10pt, top_margin=10pt, bottom_margin=42pt )

heatmap!( plt, vals; c=cmap, clims=(0,1),
    colorbar=false, xtick=false, xticks=false,
    framestyle=:box, mirror=true,
    yticks=( [1, round(Int,n/4), round(Int,n/2), round(Int,3n/4), n],
              ["0.0","0.25","0.50","0.75","1.0"] ) )

plot!( plt; ylabel="\ndeterminism, "*L"ς", yguidefont=font("Computer Modern", 10, rotation=0) )

# saveplot( plt, figurefolder*"determinism-vs-rho_beta-tau_colorbar.png"; background=:white, dpi=600 )